# 🎯 [PRC01_M06] 계약서 조항별 위험도 하이라이트 및 조달 검토 대시보드 구현
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC01/prc01_m06_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC01_M06 |
| **미션 주제** | 계약서 조항별 위험도 하이라이트 및 조달 검토 대시보드 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
조달계획안과 계약서는 지체상금률, 선금, 보증, 하도급, 필수 조항 같은 항목이 규정에 맞는지 사람이 한 건씩 읽고 대조해야 해서 검토가 늦어지고 불공정 조항이나 납기 지체 위험 문구를 놓치기 쉽다. 본 프로젝트는 조달계획안과 계약서의 오류와 위험을 검증하는 도구를 만드는 과제로, 조달 업무의 병목과 위험 신호 기준, 조항 수정 대안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 계약 자료로 계약 데이터 스키마 정의, 서식별 자료 정제와 조항 정규화, 룰 기반 규정 준수 판별, 자연어처리 기반 독소·납기 위험 문구 식별, 위험 점수와 신뢰도 엔진, 조항 하이라이트 대시보드와 오류 처리, 계약관·법무 검토 HITL 워크플로우, 1,000건 일괄 스크리닝 부하 시험, AAR 룰셋 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 계약을 근거와 함께 먼저 보여 주어 검토 시간을 줄이되, 규정 위반 여부의 판단과 계약 승인은 계약관·법무·승인권자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 📁 프로젝트 표준 산출물 디렉터리 체계 (`docs/`, `src/`, `dist/`) 안내
> **💡 [실무 프로토타입 업무 도구 구축 가이드]**
> 본 프로젝트는 단순 일회성 데이터 분석 실습이 아닌, 실제 군 부서(군수사령부, 조달본부, 계약관리과 등)에 배포 가능한 **업무 자동화 소프트웨어 도구(Tool)**를 단계별로 조립·완성하는 과정입니다. 실습 중 산출물(Deliverables)로 요구되는 파일들은 아래 표준 디렉터리에 모듈별로 저장·누적됩니다.
> - **`docs/` (국방 행정 및 기술 명세서):** 업무 정의서(`prc01_as_is_to_be_spec.md`), 스키마 명세서(`schema_definition.md`), UAT 검증 보고서(`prc01_uat_feedback_report.md`), 7개 장 운영 매뉴얼(`docs/prc01_final_manual.md`)
> - **`src/` (업무 도구 소스코드):** 데이터 파서(`connectors/`), 규정 준수 룰 엔진(`engine/`), NLP 지능 모듈(`ai/`), 대시보드 UI(`ui/`), 결함 검증기(`utils/`), HITL 워크플로우(`workflow/`), 배치 러너(`pipeline/`)
> - **`dist/` (배포 패키지):** 원클릭 자동 설치 스크립트(`dist/prc01_package/setup.sh`), 설치 가이드(`README.md`)
> *(주피터 노트북 환경에서는 각 미션 셀 검증 후 `%%writefile src/...` 매직 명령어를 활용하여 해당 모듈 파일로 자동 추출할 수 있습니다.)*

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 06. 계약서 조항별 위험도 하이라이트 및 조달 검토 대시보드 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

계약서 조항별 위험도 하이라이트 및 조달 검토 대시보드를 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **계약서 조항별 위험도 하이라이트와 조달 검토 대시보드를 구현하고 입력 검증기·에러 핸들러로 예외 상황에서도 크래시 없이 동작하게 한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **계약 방식 x 유형 위험 지도와 위험 문구가 많은 계약의 조항별 하이라이트(독소·납기위험 표시, 위험 표현 강조)를 보여 주는 대시보드 작성하여 원인을 분석하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/utils/prc01_validator.py 및 에러 핸들러` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 사용자에게 알기 쉬운 에러 메시지 및 조치 가이드 제공

#### 미션 PASS 조건

- 정량 기준: 예외 상황 발생 시 크래시(Crash) 0건, 에러 로그 기록률 100%


- 진행 기준: 예외 주입 테스트 10종 통과 및 감사 로그 파일 생성


- 검증 방법: 비정상 데이터(결측, 특수문자, 대용량) 스트레스 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""계약서 조항별 위험도 하이라이트와 계약 방식 x 유형 위험 지도를 보여 주는 조달 검토 대시보드에 입력 검증기·에러 핸들러·재시도·롤백을 붙이고, 깨진 입력 20종(결측·특수문자·대용량·일시 오류 포함)을 주입해 크래시 0건·에러 로그 기록률 100%·사용자용 조치 안내를 확인한다."""
import csv
import io
import json
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 조달계획안·계약서 생성기 (실제 계약·업체·법령 조문 없음)
# 계약 속성 16개 + 조항 문장 6개를 만들고, 속성 위반·독소 조항·납기 위험 문구가 섞인 '규정 위반 여부'를 함께 만든다.
# ---------------------------------------------------------------------------
TYPES = ["물품", "용역", "공사"]
METHODS = ["일반경쟁", "제한경쟁", "지명경쟁", "수의계약"]
CLASSES = ["정상", "독소", "납기위험"]
OBJ = ["납품 물품", "용역 결과물", "시공 목적물", "정비 성과물"]

N_TPL = [
    "계약상대자가 납기를 지체한 경우 지체일수 1일마다 계약금액의 {p}/1000에 해당하는 지체상금을 부과한다.",
    "지체상금은 불가항력 사유가 확인된 경우에 한하여 발주기관의 승인을 받아 면제할 수 있다.",
    "납기 연장은 발주기관이 불가항력 사유를 확인하고 서면으로 승인한 경우에만 인정한다.",
    "발주기관은 납품된 {o}을 {d}일 이내에 검사하고 합격한 후에 대금을 지급한다.",
    "계약상대자는 하도급을 하려는 경우 사전에 발주기관의 서면 승인을 받아야 한다.",
    "어느 한쪽이 계약을 해지하려면 사유를 명시해 {d}일 전에 서면으로 통지해야 한다.",
    "{o}에 하자가 발생하면 계약상대자는 보증 기간 안에 무상으로 보수하거나 교체한다.",
    "계약 이행 중 알게 된 비공개 사항은 제3자에게 누설하지 않으며 위반 시 책임을 진다.",
    "{o}에 대한 지식재산권의 귀속과 사용 범위는 별도 합의서에 따라 정한다.",
    "대금은 검사 합격 후 청구서를 접수한 날부터 {d}일 이내에 지급한다.",
    "계약금액의 조정은 물가변동 기준에 따라 양 당사자가 서면으로 합의한 경우에만 한다.",
    "계약상대자는 이행보증서를 계약 체결 시 제출하며 보증금은 계약 이행 완료 후 반환한다.",
    "납품 일정은 계약서 별표의 일자를 따르며 변경 시 서면 합의가 필요하다.",
    "지체상금은 면제 사유가 없는 한 부과하며 이의는 정해진 절차로 제기할 수 있다.",
    "검사에 합격하지 못한 {o}은 계약상대자가 교체하며 그 비용은 계약상대자가 부담한다.",
    "계약상대자의 귀책 사유로 이행이 불가능하면 발주기관은 서면으로 통지하고 계약을 해지할 수 있다.",
    "지연이 예상되면 계약상대자는 사유와 만회 계획을 발주기관에 서면으로 보고한다.",
    "보증 기간 중 발생한 결함은 계약상대자가 책임지고 보수한다.",
    "분쟁은 먼저 협의로 해결하고 협의가 되지 않으면 관할 법원의 판단에 따른다.",
]
T_TPL = [
    "계약상대자의 지체상금은 전액 면제하며 발주기관은 이에 대해 이의를 제기하지 못한다.",
    "납품 즉시 검사에 합격한 것으로 보며 발주기관은 이후 검사를 요구하지 않는다.",
    "계약상대자는 언제든지 일방적으로 계약을 해지할 수 있으며 그에 따른 손해는 발주기관이 부담한다.",
    "{o}의 추가 비용은 사유를 불문하고 발주기관이 전액 부담하며 별도 협의 없이 청구할 수 있다.",
    "{o}에 관한 지식재산권은 모두 계약상대자에게 귀속되고 발주기관은 사용할 수 없다.",
    "하도급은 신고 없이 제한 없이 허용하며 발주기관은 하도급 내용을 확인하지 않는다.",
    "대금은 검사 전에 전액 선지급하며 발주기관은 반환 청구권을 포기한다.",
    "계약 이행 중 알게 된 사항의 누설로 인한 책임을 계약상대자는 지지 않는다.",
    "하자 보수 책임은 인도 후 즉시 소멸하며 발주기관은 보수를 요구할 수 없다.",
    "발주기관은 계약상대자가 청구하는 금액을 거절할 수 없으며 거절하면 위약금을 지급한다.",
    "분쟁이 생기면 계약상대자가 정하는 중재 기관에서만 해결하고 발주기관은 따른다.",
    "발주기관의 검사는 생략하고 계약상대자의 시험 결과서로 대신한다.",
    "견적 금액은 사후 정산 없이 그대로 확정하고 증빙 자료는 받지 않는다.",
]
D_TPL = [
    "납기는 자재 수급 사정에 따라 계약상대자가 연장할 수 있다.",
    "납품 일정은 가능한 한 지키도록 노력하며 지연되어도 사유를 보고하지 않는다.",
    "공급 지연이나 선행 업체 사정도 불가항력으로 보아 납기를 자동 연장한다.",
    "착수일은 선행 업체의 납품이 끝난 뒤 별도로 정하며 완료 기한은 정하지 않는다.",
    "납기일 이후 1회에 한해 기한 없이 자동 연장되는 것으로 한다.",
    "납기는 계약상대자의 생산 일정이 확정되는 대로 통보한다.",
    "납품 시점은 계약상대자 사정이 허락하는 때로 하고 날짜는 따로 정하지 않는다.",
    "자재가 늦게 들어오면 완료 일정은 그만큼 자동으로 늦춰진다.",
    "기한 내에 인도하지 못해도 계약상대자에게 책임을 묻지 않는다.",
    "일정이 늦어져도 계약상대자는 새 계획서를 낼 필요가 없다.",
]
# 학습에 쓰지 않는 '처음 보는 표현'(모델이 문장 틀을 외웠는지 확인하는 용도)
HOLD = {
    "정상": ["검사에 불합격한 경우 계약상대자는 지정 기한 안에 재납품하며 재검사 비용은 상대자가 부담한다.",
           "계약상대자가 법령을 위반하여 계약을 이행하지 못하면 발주기관은 서면 통지 후 계약을 해지할 수 있다.",
           "납기가 지연될 우려가 있으면 상대자는 즉시 사유와 만회 계획을 서면으로 보고해야 한다.",
           "하자담보 책임 기간은 인수일부터 기산하며 기간 중 발생한 결함은 상대자가 책임진다."],
    "독소": ["발주기관은 계약상대자가 요구하는 대금을 이유 없이 거절하지 못하며 거절 시 위약금을 낸다.",
           "계약 내용에 관한 분쟁은 계약상대자가 지정하는 기관에서만 다루며 발주기관은 이에 따른다.",
           "발주기관의 검수 권한은 포기하고 계약상대자의 자체 시험 성적서로 갈음한다.",
           "계약상대자가 제출한 견적은 사후 정산 없이 확정 단가로 하며 증빙은 요구하지 않는다."],
    "납기위험": ["납품 시기는 상대자의 형편이 허락하는 시점으로 하고 별도 날짜는 정하지 않는다.",
             "원자재 확보가 늦어지면 완료 일자는 자동으로 뒤로 미뤄지는 것으로 본다.",
             "인도 기한은 참고용이며 지키지 못해도 책임을 묻지 않는다.",
             "일정 지연이 있어도 상대자는 변경 계획을 제출할 의무가 없다."],
}
SUFFIX = ["", "", "", " 세부 사항은 시행 세칙에 따른다.", " 이 조항은 계약 전체에 적용한다.", " 필요한 경우 양 당사자가 협의한다.", " 관련 서식은 별지를 따른다."]
SWAP = [("발주기관", "계약담당관"), ("계약상대자", "업체")]


def make_clause(rng, cls, hold=False, idx=None):
    """조항 한 문장 생성. 표현 일부는 동의어로 바꾸고 접미 문구를 무작위로 붙인다(idx를 주면 그 번호의 문장 틀을 사용)."""
    if idx is not None:
        tpl = {"정상": N_TPL, "독소": T_TPL, "납기위험": D_TPL}[cls]
        s = tpl[idx].format(p=rng.choice(["1.0", "1.5", "2.0"]), d=int(rng.choice([7, 10, 14, 30])), o=rng.choice(OBJ))
    elif hold:
        s = HOLD[cls][rng.integers(len(HOLD[cls]))]
    else:
        tpl = {"정상": N_TPL, "독소": T_TPL, "납기위험": D_TPL}[cls]
        s = tpl[rng.integers(len(tpl))].format(p=rng.choice(["1.0", "1.5", "2.0"]), d=int(rng.choice([7, 10, 14, 30])), o=rng.choice(OBJ))
    if rng.random() < 0.3:
        a, b = SWAP[rng.integers(2)]
        s = s.replace(a, b)
    return s + SUFFIX[rng.integers(len(SUFFIX))]


def make_contracts(n=200, seed=7):
    """가상 계약 n건(속성 16개 + 조항 6개 + 규정 위반 정답)을 만든다."""
    rng = np.random.default_rng(seed)
    sl = rng.beta(1.2, 3.2, n)                                  # 계약서 작성이 허술한 정도(잠재 변수)
    ctype = rng.choice(TYPES, n, p=[.5, .3, .2])
    amount = np.clip(np.exp(rng.normal(np.log(220), 1.1, n)), 5, 5000).round(0)
    method = np.where(rng.random(n) < .06 + .3 * sl, "수의계약", rng.choice(METHODS[:3], n, p=[.5, .3, .2]))
    period = np.array([rng.integers(60, 300) if t == "물품" else rng.integers(90, 400) if t == "용역" else rng.integers(180, 720) for t in ctype])
    delivery = (period * rng.uniform(.5, 1.0, n)).astype(int)
    bad_logic = rng.random(n) < .015 * (1 + 3 * sl)
    delivery = np.where(bad_logic, period + rng.integers(10, 60, n), delivery)
    pen = np.where(rng.random(n) < .22 * sl, rng.choice([.3, .5, .8], n), rng.choice([1.0, 1.5, 2.0, 2.5], n))
    adv = np.where(rng.random(n) < .22 * sl, rng.choice([50, 60, 70], n), rng.choice([0, 10, 20, 30, 40], n))
    bond = np.where(rng.random(n) < .25 * sl + .5 * (adv > 40), rng.choice([0, 5], n), rng.choice([10, 10, 15, 20], n))
    war = np.array([rng.choice([3, 6]) if (rng.random() < .25 * s and t == "물품") else rng.choice([12, 24, 36]) if t == "물품" else rng.choice([0, 6, 12]) if t == "용역" else rng.choice([24, 36, 60]) for t, s in zip(ctype, sl)])
    insp = np.where(rng.random(n) < .3 * sl, rng.choice([21, 30], n), rng.choice([5, 7, 10, 14], n))
    sub = np.where(rng.random(n) < .15 * sl, rng.choice([60, 70, 80], n), rng.choice([0, 10, 20, 30, 40, 50], n))
    padj = rng.random(n) < .75 - .4 * sl
    term = rng.random(n) > .015 * (1 + 6 * sl)
    sec = rng.random(n) > .015 * (1 + 6 * sl)
    ip = rng.random(n) > .03 * (1 + 4 * sl)
    clauses, cls_true = [], []
    for i in range(n):
        pt, pd_ = .01 + .08 * sl[i], .02 + .08 * sl[i]
        cl = [rng.choice(CLASSES, p=[1 - pt - pd_, pt, pd_]) for _ in range(6)]
        cls_true.append(cl); clauses.append([make_clause(rng, c) for c in cl])
    df = pd.DataFrame({"contract_id": [f"C-{i + 1:04d}" for i in range(n)], "ctype": ctype, "method": method, "amount_m": amount,
                       "period_days": period, "delivery_days": delivery, "penalty_permil": pen, "advance_pct": adv, "perf_bond_pct": bond,
                       "warranty_months": war, "inspection_days": insp, "subcontract_pct": sub, "price_adjust": padj,
                       "termination": term, "security": sec, "ip_clause": ip})
    hard = ((pen < 1.0) | ((adv > 40) & (bond < 10)) | ((amount >= 300) & (method == "수의계약")) | ((ctype == "물품") & (war < 12))
            | ~term | ~sec | (sub > 50) | (delivery > period))
    toxic = np.array([c.count("독소") for c in cls_true])
    delay = np.array([c.count("납기위험") for c in cls_true])
    split = (method == "수의계약") & (amount >= 250) & (amount < 300) & (rng.random(n) < .6)      # 계약 쪼개기 의심(임계값 바로 아래)
    df["clauses"], df["clause_cls"] = clauses, cls_true
    df["n_toxic"], df["n_delay"] = toxic, delay
    viol = hard | (toxic > 0) | split | (delay >= 2)
    waived = hard & (toxic == 0) & (rng.random(n) < .12)                  # 계약관이 예외 사유를 인정해 위반으로 보지 않은 건(자료의 현실적 잡음)
    hidden = ~viol & (rng.random(n) < .03 + .08 * sl)                       # 속성·문구에 드러나지 않는 위반(별첨 합의 등)
    df["violation"] = (viol & ~waived) | hidden
    df["vtype"] = np.where(hidden, "기타", np.where(hard, "속성위반", np.where(toxic > 0, "독소조항", np.where(split | (delay >= 2), "기타", "-"))))
    return df

# ---- 계약 규정 준수 판별 룰 설정(YAML 형식 문자열: 규정이 바뀌면 코드가 아니라 이 설정만 고친다) ----
RULES_YAML = """
- id: R01
  name: 지체상금률 1.0/1000 미만
  field: penalty_permil
  op: "<"
  value: 1.0
  weight: 3
- id: R02
  name: 선금 비율 40% 초과
  field: advance_pct
  op: ">"
  value: 40
  weight: 2
- id: R03
  name: 선금 40% 초과인데 이행보증 10% 미만
  field: adv_no_bond
  op: "=="
  value: true
  weight: 3
- id: R04
  name: 300백만원 이상 수의계약
  field: sole_amount
  op: ">="
  value: 300
  weight: 3
- id: R05
  name: 500백만원 이상 이행보증 10% 미만
  field: bond_gap
  op: "=="
  value: true
  weight: 2
- id: R06
  name: 물품 하자보증 12개월 미만
  field: warranty_gap
  op: "=="
  value: true
  weight: 2
- id: R07
  name: 해지 조항 없음
  field: termination
  op: "=="
  value: false
  weight: 2
- id: R08
  name: 보안 조항 없음
  field: security
  op: "=="
  value: false
  weight: 3
- id: R09
  name: 지식재산 조항 없음(물품·용역)
  field: ip_missing
  op: "=="
  value: true
  weight: 1
- id: R10
  name: 하도급 비율 50% 초과
  field: subcontract_pct
  op: ">"
  value: 50
  weight: 3
- id: R11
  name: 검사 기간 14일 초과
  field: inspection_days
  op: ">"
  value: 14
  weight: 1
- id: R12
  name: 납품 기한이 계약 기간 초과
  field: logic_err
  op: "=="
  value: true
  weight: 3
- id: R13
  name: 180일 이상 계약에 물가변동 조항 없음
  field: no_padj_long
  op: "=="
  value: true
  weight: 1
- id: R14
  name: 1000백만원 이상 비경쟁 계약
  field: noncomp_big
  op: "=="
  value: true
  weight: 2
- id: R15
  name: 계약 기간 365일 초과
  field: period_days
  op: ">"
  value: 365
  weight: 1
- id: R16
  name: 불공정·납기 위험 키워드 포함
  field: kw_hits
  op: ">="
  value: 1
  weight: 3
"""
GRADE_CUT = {"고위험": 6, "주의": 3}          # 가중치 합 기준 1차 판별 등급(필요 시 설정만 바꾼다)
# [교육용가상자료] 불공정 조항·납기 지체 위험 문구 사전(실제 규정 조문이 아님)
KEYWORDS = {"독소": ["면제하며", "일방적으로", "포기", "지지 않는다", "즉시 소멸", "제한 없이", "신고 없이", "전액 부담", "별도 협의 없이", "검사 전에", "검사에 합격한 것으로", "사용할 수 없다", "증빙은 요구하지 않는다", "지정하는 기관"],
            "납기위험": ["자동 연장", "기한 없이", "정하지 않는다", "노력하며", "연장할 수 있다", "통보한다", "책임을 묻지 않는다", "의무가 없다"]}


def parse_rules(text):
    """간이 YAML 파서(목록 + key: value). 숫자·true/false·따옴표 문자열을 해석한다."""
    rules, cur = [], None
    for line in text.strip().splitlines():
        if not line.strip():
            continue
        s = line.strip()
        if s.startswith("- "):
            cur = {}; rules.append(cur); s = s[2:]
        k, v = [x.strip() for x in s.split(":", 1)]
        v = v.strip('"')
        if v in ("true", "false"):
            v = v == "true"
        else:
            try:
                v = float(v) if "." in v else int(v)
            except ValueError:
                pass
        cur[k] = v
    return rules


OPS = {"==": lambda a, b: a == b, "!=": lambda a, b: a != b, ">": lambda a, b: a > b, ">=": lambda a, b: a >= b,
       "<": lambda a, b: a < b, "<=": lambda a, b: a <= b}


def kw_count(clauses):
    """조항 문장 목록에서 독소·납기 위험 키워드가 들어 있는 조항 수."""
    return int(sum(any(k in c for k in KEYWORDS["독소"] + KEYWORDS["납기위험"]) for c in clauses))


def derive(df):
    """룰에 쓰는 파생 필드(선금+보증 부족, 수의계약 고액, 논리 오류, 키워드 적중 등)를 만든다."""
    d = df.copy()
    d["adv_no_bond"] = (d["advance_pct"] > 40) & (d["perf_bond_pct"] < 10)
    d["sole_amount"] = np.where(d["method"] == "수의계약", d["amount_m"], 0.0)       # 수의계약일 때만 금액(한도는 YAML에서 조정)
    d["bond_gap"] = (d["amount_m"] >= 500) & (d["perf_bond_pct"] < 10)
    d["warranty_gap"] = (d["ctype"] == "물품") & (d["warranty_months"] < 12)
    d["ip_missing"] = (~d["ip_clause"]) & (d["ctype"] != "공사")
    d["logic_err"] = d["delivery_days"] > d["period_days"]
    d["no_padj_long"] = (d["period_days"] >= 180) & (~d["price_adjust"])
    d["noncomp_big"] = d["method"].isin(["지명경쟁", "수의계약"]) & (d["amount_m"] >= 1000)
    d["kw_hits"] = [kw_count(c) for c in d["clauses"]] if "clauses" in d else 0
    return d


def apply_rules(rec, rules):
    """레코드 1건(dict)에 룰을 적용해 (점수, 등급, 적중 룰 id 목록)을 돌려준다. 값이 없으면 해당 룰은 건너뛴다."""
    hits = []
    for r in rules:
        v = rec.get(r["field"])
        if v is None or (isinstance(v, float) and np.isnan(v)):
            continue
        if OPS[r["op"]](v, r["value"]):
            hits.append(r)
    score = sum(r["weight"] for r in hits)
    grade = "고위험" if score >= GRADE_CUT["고위험"] else ("주의" if score >= GRADE_CUT["주의"] else "관심")
    return score, grade, [r["id"] for r in hits]

SIMPLE_KW = {"독소": ["면제", "일방", "포기", "전액", "누설", "하도급", "선지급"], "납기위험": ["연장", "자동", "노력", "통보", "기한"]}

# ---- 조항 문장 분류(정상 / 독소 / 납기위험): 글자 n-gram TF-IDF + 로지스틱 회귀 ----
def clause_set(seed, n, hold=False, mix=(.55, .25, .20)):
    """조항 문장 n개와 정답 분류를 만든다. hold=True면 학습에 쓰지 않은 '처음 보는 표현'으로 만든다."""
    rng = np.random.default_rng(seed)
    y = rng.choice(CLASSES, n, p=list(mix))
    return [make_clause(rng, c, hold) for c in y], list(y)


def fit_clause_model(seed=100, n=1200):
    """학습용 조항 문장으로 분류 모델을 학습한다."""
    X, y = clause_set(seed, n)
    return make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True), LogisticRegression(C=20, max_iter=2000)).fit(X, y)


def keyword_pred(s):
    """비교용 기준선: 단순 키워드(부정 표현·문맥을 보지 않음) 적중으로만 분류."""
    if any(k in s for k in SIMPLE_KW["독소"]):
        return "독소"
    return "납기위험" if any(k in s for k in SIMPLE_KW["납기위험"]) else "정상"

NUM = ["amount_m", "period_days", "delivery_days", "penalty_permil", "advance_pct", "perf_bond_pct", "warranty_months", "inspection_days", "subcontract_pct"]
BOOL = ["price_adjust", "termination", "security", "ip_clause"]
NEED = ["contract_id", "ctype", "method"] + NUM + BOOL + ["clauses"]
RANGE = {"amount_m": (1, 100000), "period_days": (1, 1500), "delivery_days": (1, 1500), "penalty_permil": (0, 10), "advance_pct": (0, 100), "perf_bond_pct": (0, 100),
         "warranty_months": (0, 120), "inspection_days": (1, 90), "subcontract_pct": (0, 100)}
LOG, COMMITTED = [], {}
RULES = parse_rules(RULES_YAML)
CM = fit_clause_model()
MAX_CLAUSE = 600


class UserError(Exception):
    """사용자에게 보여 줄 메시지와 조치 가이드를 함께 가진 예외."""

    def __init__(self, code, msg, action):
        super().__init__(msg); self.code, self.msg, self.action = code, msg, action


class Transient(Exception):
    """일시 오류(재시도 대상)."""


def log(level, code, detail):
    LOG.append({"seq": len(LOG) + 1, "level": level, "code": code, "detail": str(detail)[:70]})


def read_input(src):
    """입력(CSV 문자열 / JSON 문자열 / DataFrame)을 DataFrame으로 읽는다. 실패하면 UserError."""
    if src is None:
        raise UserError("E01", "입력이 비어 있습니다.", "파일을 다시 선택해 주세요.")
    if isinstance(src, pd.DataFrame):
        df = src.copy()
    elif isinstance(src, str):
        if not src.strip():
            raise UserError("E02", "빈 파일입니다.", "내용이 있는 파일인지 확인해 주세요.")
        try:
            df = pd.DataFrame(json.loads(src)) if src.lstrip()[0] in "[{" else pd.DataFrame(list(csv.DictReader(io.StringIO(src))))
        except Exception as e:
            raise UserError("E03", "파일 형식을 읽을 수 없습니다.", "CSV(쉼표 구분) 또는 JSON(목록) 형식으로 저장했는지 확인해 주세요.") from e
    else:
        raise UserError("E04", f"지원하지 않는 입력 형식입니다({type(src).__name__}).", "CSV 또는 JSON 파일을 사용해 주세요.")
    if df.empty:
        raise UserError("E02", "읽은 행이 0건입니다.", "머리글만 있는 파일이 아닌지 확인해 주세요.")
    return df


def clean_text(s):
    """특수문자·제어문자 제거, 지나치게 긴 조항은 잘라 내기."""
    s = re.sub(r"[\x00-\x08\x0b-\x1f\x7f<>]", " ", str(s))
    return re.sub(r"\s+", " ", s).strip()[:MAX_CLAUSE]


def validate_table(df):
    """열 점검 -> 행 단위 점검. 문제 행은 격리하고 나머지로 계속 진행한다. 반환: (정상 행, 격리 내역)"""
    miss = [c for c in NEED if c not in df.columns]
    if miss:
        raise UserError("E10", f"필수 열이 없습니다: {', '.join(miss[:4])}{' 외' if len(miss) > 4 else ''}", "열 이름이 양식(데이터 딕셔너리)과 같은지 확인해 주세요.")
    d, bad = df.copy().reset_index(drop=True), {}
    for c in NUM:
        conv = pd.to_numeric(d[c], errors="coerce")
        for i in d.index[conv.isna()]:
            bad.setdefault(i, f"{c}: 숫자가 아님/누락")
        d[c] = conv
    for c, (lo, hi) in RANGE.items():
        for i in d.index[(d[c] < lo) | (d[c] > hi)]:
            bad.setdefault(i, f"{c}: 허용 범위({lo}~{hi}) 밖")
    for c, ok in (("ctype", TYPES), ("method", METHODS)):
        for i in d.index[~d[c].isin(ok)]:
            bad.setdefault(i, f"{c}: 허용되지 않은 값")
    for i in d.index[d["contract_id"].duplicated(keep="first")]:
        bad.setdefault(i, "contract_id: 중복")
    for i in d.index[~d["contract_id"].astype(str).str.match(r"^C-\d{4}$")]:
        bad.setdefault(i, "contract_id: 형식 오류")
    for c in BOOL:
        d[c] = d[c].map(lambda v: v if isinstance(v, (bool, np.bool_)) else str(v).strip().lower() in ("true", "1", "예", "y"))
    d["clauses"] = d["clauses"].map(lambda v: [clean_text(x) for x in (v if isinstance(v, list) else str(v).split(" || ")) if str(x).strip()])
    for i in d.index[d["clauses"].map(len) == 0]:
        bad.setdefault(i, "clauses: 조항 없음")
    if len(bad) == len(d):
        raise UserError("E11", "사용할 수 있는 행이 없습니다.", "격리 사유를 확인해 값을 고친 뒤 다시 올려 주세요.")
    return d.drop(index=list(bad)).reset_index(drop=True), bad


MARK = {"독소": "[!!]", "납기위험": "[! ]", "정상": "[ok]"}


def highlight(clauses):
    """조항별 위험 표시: 모델 분류 결과와 키워드 위치를 【 】로 강조한다."""
    cls = CM.predict(clauses)
    out = []
    for c, k in zip(clauses, cls):
        for kw in KEYWORDS["독소"] + KEYWORDS["납기위험"]:
            if k != "정상" and kw in c:
                c = c.replace(kw, f"【{kw}】")
        out.append((k, c))
    return out


def build_dashboard(df, fail_times=0, state=None):
    """대시보드 산출물을 한 번에 만든다(전부 성공해야 반영 = 트랜잭션). 반환: 결과 dict, 격리 내역"""
    d, bad = validate_table(read_input(df))
    if state is not None and state["calls"] < fail_times:
        state["calls"] += 1
        raise Transient("외부 저장소 응답 지연(모의)")
    dd = derive(d)
    dd["score"] = [apply_rules(r, RULES)[0] for r in dd.to_dict("records")]
    heat = dd.pivot_table(index="method", columns="ctype", values="score", aggfunc="mean").round(1)
    top = dd.sort_values(["kw_hits", "score"], ascending=False).head(3)       # 위험 문구가 많은 계약부터
    hl = {r["contract_id"]: highlight(r["clauses"]) for _, r in top.iterrows()}
    return {"heat": heat, "top": top[["contract_id", "ctype", "method", "amount_m", "score"]], "hl": hl, "n": len(dd)}, bad


def safe_dashboard(case, src, fail_times=0, always_fail=False):
    """모든 예외를 잡아 크래시 없이 사용자 메시지·조치 안내를 돌려주고 에러 로그를 남긴다. 일시 오류는 3회까지 재시도하고, 끝내 실패하면 직전 정상 결과로 롤백한다."""
    state = {"calls": 0}
    for attempt in range(1, 4):
        try:
            if always_fail:
                raise Transient("저장소 연결 불가(모의)")
            res, bad = build_dashboard(src, fail_times, state)
            for i, why in bad.items():
                log("WARN", "W01", f"{case}: 행 {i} 격리 - {why}")
            COMMITTED["last"] = res                                          # 전부 성공했을 때만 반영
            return {"case": case, "ok": True, "msg": f"표시 완료(격리 {len(bad)}행)" if bad else "표시 완료" + (f", 재시도 {attempt - 1}회 후 성공" if attempt > 1 else ""), "action": "-"}
        except UserError as e:
            log("ERROR", e.code, f"{case}: {e.msg}")
            return {"case": case, "ok": False, "msg": e.msg, "action": e.action}
        except Transient as e:
            log("WARN", "W02", f"{case}: 일시 오류 시도 {attempt}/3 - {e}")
            time.sleep(0.01)
        except Exception as e:                                               # 예상하지 못한 오류도 크래시 대신 안내로 변환
            log("ERROR", "E99", f"{case}: {type(e).__name__} {e}")
            return {"case": case, "ok": False, "msg": "예상하지 못한 오류가 발생했습니다.", "action": "오류 코드 E99와 입력 파일을 담당자에게 전달해 주세요."}
    log("ERROR", "E50", f"{case}: 재시도 3회 실패 -> 직전 정상 결과로 롤백")
    return {"case": case, "ok": False, "msg": "일시적인 문제로 저장하지 못했습니다. 직전 정상 화면을 유지합니다.", "action": "잠시 후 다시 시도해 주세요."}


con = make_contracts(100, seed=7)
base = con.assign(clauses=con["clauses"].map(lambda c: " || ".join(c)))            # CSV처럼 조항을 한 칸에 이어 붙인 형태
csv_ok = base.drop(columns=["clause_cls", "n_toxic", "n_delay", "violation", "vtype"]).to_csv(index=False)


def mut(**kw):
    d = base.copy()
    for k, v in kw.items():
        d[k] = d[k].astype(object); d.loc[d.index[:3], k] = v
    return d


big = pd.concat([base] * 200, ignore_index=True)
big["contract_id"] = [f"C-{i % 10000:04d}" for i in range(len(big))]               # 대용량(20,000행, 번호 중복은 격리 대상)
cases = [
    ("정상 입력", base, {}), ("정상 CSV 문자열", csv_ok, {}), ("입력 없음(None)", None, {}), ("빈 문자열", "  ", {}), ("깨진 JSON", '[{"contract_id": "C-0001", ', {}),
    ("지원하지 않는 형식(정수)", 12345, {}), ("머리글만 있는 CSV", ",".join(base.columns) + "\n", {}), ("필수 열 누락", base.drop(columns=["penalty_permil", "clauses"]), {}),
    ("숫자 열에 문자", mut(amount_m="미정"), {}), ("음수 금액", mut(amount_m=-5), {}), ("계약 방식 결측", mut(method=np.nan), {}), ("허용되지 않은 유형", mut(ctype="기타"), {}),
    ("계약번호 중복·형식 오류", mut(contract_id="C-9"), {}), ("조항 칸이 비어 있음", mut(clauses=""), {}),
    ("특수문자·제어문자·태그", mut(clauses="<script>alert(1)</script> \x00\x07 지체상금은 전액 면제하며"), {}),
    ("조항 문장 1MB", mut(clauses="가" * 1_000_000), {}), ("극단값(1e12)", mut(amount_m=1e12), {}),
    ("대용량 20,000행", big, {}), ("일시 오류 2회 후 성공", base, {"fail_times": 2}), ("끝내 실패(롤백)", base, {"always_fail": True}),
]
res, crash, times = [], 0, {}
for name, src, kw in cases:
    t0 = time.perf_counter()
    try:
        res.append(safe_dashboard(name, src, **kw))
    except BaseException:                                                    # safe_dashboard 밖으로 예외가 새 나간 건수
        crash += 1
    times[name] = time.perf_counter() - t0
print(f"[에러 핸들러 시험] 입력 {len(cases)}종 실행")
for r in res:
    print(f"  {'OK ' if r['ok'] else '안내'} {r['case']:22s} -> {r['msg'][:46]}" + (f" | 조치: {r['action'][:34]}" if r["action"] != "-" else ""))
n_err = sum(not r["ok"] for r in res)
logged_err = sum(l["level"] == "ERROR" for l in LOG)
print(f"\n예외 주입 {len(cases)}종 실행 완료(목표 10종 이상: {'달성' if len(cases) >= 10 else '미달'}), 크래시 {crash}건(목표 0건: {'달성' if crash == 0 else '미달'})")
print(f"오류 {n_err}건 중 ERROR 로그 {logged_err}건 -> 기록률 {min(logged_err / max(n_err, 1), 1):.0%}(목표 100%: {'달성' if logged_err == n_err else '미달'}), 안내 문구에 조치 가이드 포함 {sum(r['action'] != '-' for r in res if not r['ok'])}/{n_err}건")
print(f"행 격리 경고 {sum(l['code'] == 'W01' for l in LOG)}건, 재시도 경고 {sum(l['code'] == 'W02' for l in LOG)}건 기록, 대용량 20,000행 처리 {times['대용량 20,000행']:.1f}초")
print("직전 정상 결과 유지(롤백): 마지막 반영된 화면의 계약 수 =", COMMITTED["last"]["n"], "건")
print("\n[에러 로그 앞 5건]")
print(pd.DataFrame(LOG).head(5).to_string(index=False))

# 정상 입력으로 대시보드(조항 하이라이트 + 위험 지도) 보여 주기
out, bad = build_dashboard(base)
print("\n[위험 지도] 계약 방식 x 유형 평균 룰 점수")
print(out["heat"].fillna(0).to_string())
print("\n[조항별 위험 하이라이트] 위험 문구가 많은 상위 3건 (【 】= 위험 표현, [!!]=독소, [! ]=납기위험, [ok]=정상)")
for _, r in out["top"].iterrows():
    print(f"  {r['contract_id']} {r['ctype']} {r['method']} {r['amount_m']:.0f}백만원 룰 점수 {r['score']}")
    for k, c in out["hl"][r["contract_id"]]:
        print(f"     {MARK[k]} {c[:52]}")
print("※ 하이라이트는 검토 우선순위 표시이며 불공정 여부의 최종 판단은 계약관·법무 담당자가 한다.")

heat = out["heat"].fillna(0)
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
im = ax[0].imshow(heat.to_numpy(), cmap="Reds"); ax[0].set_xticks(range(heat.shape[1]), ["goods", "service", "works"][:heat.shape[1]]); ax[0].set_yticks(range(heat.shape[0]), [f"m{i + 1}" for i in range(heat.shape[0])])
ax[0].set_title("Risk map (method x type)"); plt.colorbar(im, ax=ax[0])
ax[1].bar(["ok", "guided", "crash"], [sum(r["ok"] for r in res), n_err, crash], color=["tab:green", "gold", "tab:red"]); ax[1].set_title("Error injection results (20 cases)")
cnt = pd.Series([k for _, cl in out["hl"].items() for k, _ in cl]).value_counts()
ax[2].bar([{"독소": "toxic", "납기위험": "delay", "정상": "ok"}[k] for k in cnt.index], cnt.values, color="tab:blue"); ax[2].set_title("Clause labels in top-3 contracts")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 06 예상 실행 결과 (Output)</b></summary>

```text
[에러 핸들러 시험] 입력 20종 실행
  OK  정상 입력                  -> 표시 완료
  OK  정상 CSV 문자열             -> 표시 완료
  안내 입력 없음(None)            -> 입력이 비어 있습니다. | 조치: 파일을 다시 선택해 주세요.
  안내 빈 문자열                  -> 빈 파일입니다. | 조치: 내용이 있는 파일인지 확인해 주세요.
  안내 깨진 JSON                -> 파일 형식을 읽을 수 없습니다. | 조치: CSV(쉼표 구분) 또는 JSON(목록) 형식으로 저장했는지 
  안내 지원하지 않는 형식(정수)         -> 지원하지 않는 입력 형식입니다(int). | 조치: CSV 또는 JSON 파일을 사용해 주세요.
  안내 머리글만 있는 CSV            -> 읽은 행이 0건입니다. | 조치: 머리글만 있는 파일이 아닌지 확인해 주세요.
  안내 필수 열 누락                -> 필수 열이 없습니다: penalty_permil, clauses | 조치: 열 이름이 양식(데이터 딕셔너리)과 같은지 확인해 주세요.
  OK  숫자 열에 문자               -> 표시 완료(격리 3행)
  OK  음수 금액                  -> 표시 완료(격리 3행)
  OK  계약 방식 결측               -> 표시 완료(격리 3행)
  OK  허용되지 않은 유형             -> 표시 완료(격리 3행)
  OK  계약번호 중복·형식 오류          -> 표시 완료(격리 3행)
  OK  조항 칸이 비어 있음            -> 표시 완료(격리 3행)
  OK  특수문자·제어문자·태그           -> 표시 완료
  OK  조항 문장 1MB              -> 표시 완료
  OK  극단값(1e12)              -> 표시 완료(격리 3행)
  OK  대용량 20,000행            -> 표시 완료(격리 10000행)
  OK  일시 오류 2회 후 성공          -> 표시 완료, 재시도 2회 후 성공
  안내 끝내 실패(롤백)              -> 일시적인 문제로 저장하지 못했습니다. 직전 정상 화면을 유지합니다. | 조치: 잠시 후 다시 시도해 주세요.

예외 주입 20종 실행 완료(목표 10종 이상: 달성), 크래시 0건(목표 0건: 달성)
오류 7건 중 ERROR 로그 7건 -> 기록률 100%(목표 100%: 달성), 안내 문구에 조치 가이드 포함 7/7건
행 격리 경고 10021건, 재시도 경고 5건 기록, 대용량 20,000행 처리 2.6초
직전 정상 결과 유지(롤백): 마지막 반영된 화면의 계약 수 = 100 건

[에러 로그 앞 5건]
 seq level code                                 detail
   1 ERROR  E01              입력 없음(None): 입력이 비어 있습니다.
   2 ERROR  E02                        빈 문자열: 빈 파일입니다.
   3 ERROR  E03             깨진 JSON: 파일 형식을 읽을 수 없습니다.
   4 ERROR  E04 지원하지 않는 형식(정수): 지원하지 않는 입력 형식입니다(int).
   5 ERROR  E02              머리글만 있는 CSV: 읽은 행이 0건입니다.

[위험 지도] 계약 방식 x 유형 평균 룰 점수
ctype    공사   물품   용역
method               
수의계약    1.0  2.5  2.3
일반경쟁    2.7  1.8  1.1
... (이하 생략)
```
</details>
